## Instala e atualiza as bibliotecas necessárias para QLoRA; após executar, é necessário reiniciar a sessão.

In [ ]:
%pip install -q --upgrade \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes

##  Exibe as GPUs disponibilizadas pelo Kaggle, incluindo modelo, memória, driver e versão CUDA suportada.

In [1]:
!nvidia-smi

Mon Sep  7 13:56:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
"""
Limita o treinamento à primeira GPU e desativa o 
paralelismo adicional do tokenizer.
"""

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
"""
Recupera o token do Hugging Face e o disponibiliza às bibliotecas internas.
"""
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token

print("HF_TOKEN configurado:", bool(hf_token))

HF_TOKEN configurado: True


In [4]:
"""
Valida a integração entre PyTorch e CUDA e determina a precisão 
recomendada para a GPU disponível.
"""

import torch

print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("CUDA disponível:", torch.cuda.is_available())
print("Quantidade de GPUs visíveis:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU selecionada:", torch.cuda.get_device_name(0))

    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM disponível: {total_vram:.2f} GB")

    if torch.cuda.get_device_capability(0)[0] >= 8:
        compute_dtype = torch.bfloat16
    else:
        compute_dtype = torch.float16

    print("Precisão recomendada:", compute_dtype)

PyTorch: 2.10.0+cu128
CUDA do PyTorch: 12.8
CUDA disponível: True
Quantidade de GPUs visíveis: 1
GPU selecionada: Tesla T4
VRAM disponível: 14.56 GB
Precisão recomendada: torch.float16


In [5]:
"""
Localiza e exibe os arquivos JSONL anexados ao 
notebook como dados de entrada.
"""

from pathlib import Path

jsonl_files = sorted(Path("/kaggle/input").rglob("*.jsonl"))

for file_path in jsonl_files:
    print(file_path)

/kaggle/input/datasets/dharmasistemas/medassist-medquad-splits/test.jsonl
/kaggle/input/datasets/dharmasistemas/medassist-medquad-splits/train.jsonl
/kaggle/input/datasets/dharmasistemas/medassist-medquad-splits/validation.jsonl


In [6]:
"""
Lê e valida manualmente os arquivos JSONL, contabilizando 
os registros de cada split.
"""

import json
from pathlib import Path

DATA_DIR = Path(
    "/kaggle/input/datasets/dharmasistemas/medassist-medquad-splits"
)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALIDATION_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"

files = {
    "train": TRAIN_FILE,
    "validation": VALIDATION_FILE,
    "test": TEST_FILE,
}

datasets_raw = {}

for split_name, file_path in files.items():
    records = []

    with file_path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON inválido em {file_path}, linha {line_number}"
                ) from error

    datasets_raw[split_name] = records
    print(f"{split_name}: {len(records)} registros")

train: 4795 registros
validation: 603 registros
test: 602 registros


In [7]:
"""
Inspeciona o primeiro registro de cada split para conferir campos, 
conteúdo e estrutura do dataset.
"""

for split_name, records in datasets_raw.items():
    print(f"\n--- {split_name.upper()} ---")

    if not records:
        print("Arquivo vazio")
        continue

    first_record = records[0]

    print("Campos:", list(first_record.keys()))

    for field, value in first_record.items():
        preview = str(value).replace("\n", " ")[:300]
        print(f"{field}: {preview!r}")


--- TRAIN ---
Campos: ['id', 'document_id', 'question_id', 'pair_id', 'question_type', 'focus', 'focus_synonyms', 'question', 'answer', 'source', 'url', 'collection', 'xml_path', 'umls_cuis', 'semantic_types', 'semantic_groups']
id: 'medquad_00284696ac2910fb63bc632e'
document_id: '0000059'
question_id: '0000059-8'
pair_id: '8'
question_type: 'treatment'
focus: 'Heart Block'
focus_synonyms: '[]'
question: 'What are the treatments for Heart Block ?'
answer: "Treatment depends on the type of heart block you have. If you have first-degree heart block, you may not need treatment. If you have second-degree heart block, you may need a pacemaker. A pacemaker is a small device that's placed under the skin of your chest or abdomen. This device uses electrical p"
source: 'NHLBI'
url: 'http://www.nhlbi.nih.gov/health/health-topics/topics/hb'
collection: '8_NHLBI_QA_XML'
xml_path: '8_NHLBI_QA_XML/0000059.xml'
umls_cuis: "['C0018794']"
semantic_types: "['T047']"
semantic_groups: "['Disorders']"

--

In [8]:
"""
Registra as versões das bibliotecas e confirma que o 
PyTorch reconhece a GPU selecionada.
"""

import torch
import transformers
import datasets
import accelerate
import peft
import trl
import bitsandbytes

print("PyTorch:", torch.__version__)
print("CUDA do PyTorch:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print("CUDA disponível:", torch.cuda.is_available())
print("GPUs visíveis:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA do PyTorch: 12.8
Transformers: 5.16.1
Datasets: 5.0.1
Accelerate: 1.14.0
PEFT: 0.20.0
TRL: 1.12.0
BitsAndBytes: 0.50.2
CUDA disponível: True
GPUs visíveis: 1
GPU: Tesla T4


In [9]:
"""
Carrega o tokenizer e o template conversacional oficial 
do Llama 3.2 1B Instruct.
"""

from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer carregado:", MODEL_ID)
print("Vocabulário:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Chat template disponível:", tokenizer.chat_template is not None)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer carregado: meta-llama/Llama-3.2-1B-Instruct
Vocabulário: 128256
Pad token: <|eot_id|>
EOS token: <|eot_id|>
Chat template disponível: True


In [10]:
"""
Carrega treino, validação e teste como um DatasetDict da 
biblioteca Hugging Face Datasets.
"""
from datasets import load_dataset

DATA_DIR = "/kaggle/input/datasets/dharmasistemas/medassist-medquad-splits"

raw_datasets = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/train.jsonl",
        "validation": f"{DATA_DIR}/validation.jsonl",
        "test": f"{DATA_DIR}/test.jsonl",
    },
)

print(raw_datasets)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'question_id', 'pair_id', 'question_type', 'focus', 'focus_synonyms', 'question', 'answer', 'source', 'url', 'collection', 'xml_path', 'umls_cuis', 'semantic_types', 'semantic_groups'],
        num_rows: 4795
    })
    validation: Dataset({
        features: ['id', 'document_id', 'question_id', 'pair_id', 'question_type', 'focus', 'focus_synonyms', 'question', 'answer', 'source', 'url', 'collection', 'xml_path', 'umls_cuis', 'semantic_types', 'semantic_groups'],
        num_rows: 603
    })
    test: Dataset({
        features: ['id', 'document_id', 'question_id', 'pair_id', 'question_type', 'focus', 'focus_synonyms', 'question', 'answer', 'source', 'url', 'collection', 'xml_path', 'umls_cuis', 'semantic_types', 'semantic_groups'],
        num_rows: 602
    })
})


In [11]:
"""
Verifica se existem perguntas ou respostas vazias 
nos três splits do dataset.
"""
for split_name, split in raw_datasets.items():
    empty_questions = sum(
        not str(record["question"]).strip()
        for record in split
    )

    empty_answers = sum(
        not str(record["answer"]).strip()
        for record in split
    )

    print(
        f"{split_name}: "
        f"perguntas vazias={empty_questions}, "
        f"respostas vazias={empty_answers}"
    )

train: perguntas vazias=0, respostas vazias=0
validation: perguntas vazias=0, respostas vazias=0
test: perguntas vazias=0, respostas vazias=0


In [12]:
"""
Define o prompt de sistema e converte os pares de pergunta e 
resposta para o formato conversacional.
"""

SYSTEM_PROMPT = (
    "You are MedAssist, an educational medical information assistant. "
    "Answer the user's medical question clearly and objectively. "
    "Do not present the answer as a medical diagnosis or replace professional care."
)


def convert_to_conversation(example):
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": example["question"].strip(),
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"].strip(),
            }
        ],
    }


training_datasets = raw_datasets.map(
    convert_to_conversation,
    remove_columns=raw_datasets["train"].column_names,
    desc="Convertendo para formato conversacional",
)

print(training_datasets)
print(training_datasets["train"].column_names)

Convertendo para formato conversacional:   0%|          | 0/4795 [00:00<?, ? examples/s]

Convertendo para formato conversacional:   0%|          | 0/603 [00:00<?, ? examples/s]

Convertendo para formato conversacional:   0%|          | 0/602 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 4795
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 603
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 602
    })
})
['prompt', 'completion']


In [13]:
"""
Exibe um exemplo completo após a aplicação do chat template do Llama.
"""

example = training_datasets["train"][0]

messages = example["prompt"] + example["completion"]

formatted_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

print(formatted_text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 07 Sep 2026

You are MedAssist, an educational medical information assistant. Answer the user's medical question clearly and objectively. Do not present the answer as a medical diagnosis or replace professional care.<|eot_id|><|start_header_id|>user<|end_header_id|>

What are the treatments for Heart Block ?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Treatment depends on the type of heart block you have. If you have first-degree heart block, you may not need treatment. If you have second-degree heart block, you may need a pacemaker. A pacemaker is a small device that's placed under the skin of your chest or abdomen. This device uses electrical pulses to prompt the heart to beat at a normal rate. If you have third-degree heart block, you will need a pacemaker. In an emergency, a temporary pacemaker might be used until you can get a long-term device. Most people who

In [14]:
"""
Investigação dos exemplos sobre cálculos renais que os resultados fine-tuning piloto apontaram revisão.
"""

terms = [
    "kidney stone",
    "kidney stones",
    "renal stone",
    "renal stones",
    "nephrolithiasis",
]

kidney_stone_examples = []

for split_name in ["train", "validation", "test"]:
    for record in raw_datasets[split_name]:
        searchable_text = " ".join([
            str(record.get("focus", "")),
            str(record.get("question", "")),
            str(record.get("answer", "")),
        ]).lower()

        if any(term in searchable_text for term in terms):
            kidney_stone_examples.append({
                "split": split_name,
                "id": record["id"],
                "focus": record["focus"],
                "question": record["question"],
                "answer": record["answer"],
            })

print(
    "Exemplos relacionados a cálculos renais:",
    len(kidney_stone_examples),
)

for example in kidney_stone_examples:
    print("\n" + "=" * 80)
    print("Split:", example["split"])
    print("ID:", example["id"])
    print("Focus:", example["focus"])
    print("Pergunta:", example["question"])
    print("Resposta:")
    print(example["answer"][:1500])

Exemplos relacionados a cálculos renais: 57

Split: train
ID: medquad_093e988de0ed3a04cac04171
Focus: alkaptonuria
Pergunta: What is (are) alkaptonuria ?
Resposta:
Alkaptonuria is an inherited condition that causes urine to turn black when exposed to air. Ochronosis, a buildup of dark pigment in connective tissues such as cartilage and skin, is also characteristic of the disorder. This blue-black pigmentation usually appears after age 30. People with alkaptonuria typically develop arthritis, particularly in the spine and large joints, beginning in early adulthood. Other features of this condition can include heart problems, kidney stones, and prostate stones.

Split: train
ID: medquad_1f154f0c461a66a31093df65
Focus: Sarcoidosis
Pergunta: What are the symptoms of Sarcoidosis ?
Resposta:
What are the signs and symptoms of Sarcoidosis? Many people who have sarcoidosis don't have symptoms. Others may feel like they are coming down with the flu or a respiratory infection. While almost any b

In [15]:
"""
Verifica quais exemplos relacionados a cálculos renais fizeram
parte da amostra aleatória de 500 registros do treino piloto.
"""

import numpy as np

PILOT_SEED = 42
PILOT_TRAIN_SIZE = 500

rng = np.random.default_rng(PILOT_SEED)

pilot_train_indices_check = rng.permutation(
    len(raw_datasets["train"])
)[:PILOT_TRAIN_SIZE].tolist()

pilot_train_ids_check = set(
    raw_datasets["train"].select(
        pilot_train_indices_check
    )["id"]
)

related_train_examples = [
    example
    for example in kidney_stone_examples
    if example["split"] == "train"
]

related_examples_in_pilot = [
    example
    for example in related_train_examples
    if example["id"] in pilot_train_ids_check
]

focused_examples_in_pilot = [
    example
    for example in related_examples_in_pilot
    if any(
        term in example["focus"].lower()
        for term in [
            "kidney stone",
            "renal stone",
            "nephrolithiasis",
        ]
    )
]

print(
    "Menções relacionadas no treino completo:",
    len(related_train_examples),
)

print(
    "Menções relacionadas presentes no piloto:",
    len(related_examples_in_pilot),
)

print(
    "Exemplos com cálculos renais como foco no piloto:",
    len(focused_examples_in_pilot),
)

for example in focused_examples_in_pilot:
    print("\n" + "=" * 80)
    print("ID:", example["id"])
    print("Focus:", example["focus"])
    print("Pergunta:", example["question"])

Menções relacionadas no treino completo: 43
Menções relacionadas presentes no piloto: 5
Exemplos com cálculos renais como foco no piloto: 2

ID: medquad_d33abec93d0341bbb49a1ed6
Focus: Kidney Stones in Children
Pergunta: What causes Kidney Stones in Children ?

ID: medquad_e11755717193a17738e5ab19
Focus: Kidney Stones in Children
Pergunta: What is (are) Kidney Stones in Children ?


In [16]:
"""
Prepara os conjuntos definitivos de treino e validação usando todos
os registros disponíveis, preservando o conjunto de teste exclusivamente
para a avaliação final do modelo.
"""

FINAL_SEED = 42

final_train = training_datasets["train"]
final_validation = training_datasets["validation"]

train_ids = set(raw_datasets["train"]["id"])
validation_ids = set(raw_datasets["validation"]["id"])
test_ids = set(raw_datasets["test"]["id"])

train_validation_overlap = train_ids & validation_ids
train_test_overlap = train_ids & test_ids
validation_test_overlap = validation_ids & test_ids

print("Treino definitivo:", len(final_train))
print("Validação definitiva:", len(final_validation))
print("Teste usado no treinamento: 0")

print(
    "IDs compartilhados entre treino e validação:",
    len(train_validation_overlap),
)

print(
    "IDs compartilhados entre treino e teste:",
    len(train_test_overlap),
)

print(
    "IDs compartilhados entre validação e teste:",
    len(validation_test_overlap),
)

assert len(final_train) == 4795
assert len(final_validation) == 603

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

print("Conjuntos definitivos preparados e validados.")

Treino definitivo: 4795
Validação definitiva: 603
Teste usado no treinamento: 0
IDs compartilhados entre treino e validação: 0
IDs compartilhados entre treino e teste: 0
IDs compartilhados entre validação e teste: 0
Conjuntos definitivos preparados e validados.


In [17]:
"""
Calcula a quantidade real de tokens de cada exemplo antes 
de aplicar truncamento.
"""

def count_tokens(example):
    messages = example["prompt"] + example["completion"]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    input_ids = tokenizer(
        formatted_text,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    return {
        "num_tokens": len(input_ids)
    }


datasets_with_lengths = training_datasets.map(
    count_tokens,
    desc="Calculando quantidade de tokens",
)

print(datasets_with_lengths)

Calculando quantidade de tokens:   0%|          | 0/4795 [00:00<?, ? examples/s]

Calculando quantidade de tokens:   0%|          | 0/603 [00:00<?, ? examples/s]

Calculando quantidade de tokens:   0%|          | 0/602 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion', 'num_tokens'],
        num_rows: 4795
    })
    validation: Dataset({
        features: ['prompt', 'completion', 'num_tokens'],
        num_rows: 603
    })
    test: Dataset({
        features: ['prompt', 'completion', 'num_tokens'],
        num_rows: 602
    })
})


In [18]:
"""
Analisa a distribuição dos comprimentos para justificar o limite de 
1024 tokens usado no treinamento.
"""

import numpy as np

LIMITS = [256, 512, 768, 1024]

for split_name, split in datasets_with_lengths.items():
    lengths = np.array(split["num_tokens"])

    print(f"\n--- {split_name.upper()} ---")
    print("Mínimo:", int(lengths.min()))
    print("Mediana:", int(np.percentile(lengths, 50)))
    print("Percentil 90:", int(np.percentile(lengths, 90)))
    print("Percentil 95:", int(np.percentile(lengths, 95)))
    print("Percentil 99:", int(np.percentile(lengths, 99)))
    print("Máximo:", int(lengths.max()))

    for limit in LIMITS:
        quantity = int((lengths > limit).sum())
        percentage = quantity / len(lengths) * 100

        print(
            f"Acima de {limit}: "
            f"{quantity} registros ({percentage:.2f}%)"
        )


--- TRAIN ---
Mínimo: 91
Mediana: 268
Percentil 90: 624
Percentil 95: 819
Percentil 99: 1706
Máximo: 5264
Acima de 256: 2518 registros (52.51%)
Acima de 512: 773 registros (16.12%)
Acima de 768: 291 registros (6.07%)
Acima de 1024: 153 registros (3.19%)

--- VALIDATION ---
Mínimo: 93
Mediana: 255
Percentil 90: 577
Percentil 95: 727
Percentil 99: 1459
Máximo: 4308
Acima de 256: 299 registros (49.59%)
Acima de 512: 78 registros (12.94%)
Acima de 768: 28 registros (4.64%)
Acima de 1024: 13 registros (2.16%)

--- TEST ---
Mínimo: 93
Mediana: 269
Percentil 90: 598
Percentil 95: 762
Percentil 99: 1612
Máximo: 5116
Acima de 256: 334 registros (55.48%)
Acima de 512: 91 registros (15.12%)
Acima de 768: 30 registros (4.98%)
Acima de 1024: 17 registros (2.82%)


In [19]:
"""
Carrega o modelo-base em 4 bits com NF4 e double quantization para 
reduzir o consumo de VRAM.
"""

import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
MAX_LENGTH = 1024

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

print("Modelo carregado:", MODEL_ID)
print("Carregado em 4 bits:", getattr(model, "is_loaded_in_4bit", False))
print(
    "Memória ocupada pelo modelo:",
    f"{model.get_memory_footprint() / 1024**3:.2f} GB",
)

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Modelo carregado: meta-llama/Llama-3.2-1B-Instruct
Carregado em 4 bits: True
Memória ocupada pelo modelo: 0.94 GB


In [20]:
"""
Mede a memória utilizada pelo modelo e confirma que ele 
foi carregado na GPU correta.
"""

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
free_memory, total_memory = torch.cuda.mem_get_info()

print(f"VRAM alocada pelo PyTorch: {allocated:.2f} GB")
print(f"VRAM reservada pelo PyTorch: {reserved:.2f} GB")
print(f"VRAM livre: {free_memory / 1024**3:.2f} GB")
print(f"VRAM total: {total_memory / 1024**3:.2f} GB")
print("Dispositivo do modelo:", next(model.parameters()).device)

VRAM alocada pelo PyTorch: 0.96 GB
VRAM reservada pelo PyTorch: 1.00 GB
VRAM livre: 13.45 GB
VRAM total: 14.56 GB
Dispositivo do modelo: cuda:0


In [21]:
"""
Prepara o modelo quantizado, adiciona os adaptadores LoRA e confirma
os parâmetros treináveis.
"""

from collections import Counter

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)

# A T4 usará autocast FP16, enquanto os pequenos adaptadores
# permanecem em FP32 para estabilidade e compatibilidade com GradScaler.
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

model.config.use_cache = False
_ = model.train()

trainable_dtypes = Counter(
    str(parameter.dtype)
    for parameter in model.parameters()
    if parameter.requires_grad
)

model.print_trainable_parameters()
print("Tipos dos parâmetros treináveis:", trainable_dtypes)

assert set(trainable_dtypes) == {"torch.float32"}

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540
Tipos dos parâmetros treináveis: Counter({'torch.float32': 224})


In [22]:
"""
Executa uma inferência de referência com os adaptadores 
desativados para registrar a resposta do modelo-base.
"""

test_example = raw_datasets["test"][0]

baseline_messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT,
    },
    {
        "role": "user",
        "content": test_example["question"].strip(),
    },
]

baseline_inputs = tokenizer.apply_chat_template(
    baseline_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

baseline_inputs = {
    name: tensor.to("cuda:0")
    for name, tensor in baseline_inputs.items()
}

model.eval()
model.config.use_cache = True

with model.disable_adapter():
    with torch.inference_mode():
        baseline_output = model.generate(
            **baseline_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

prompt_length = baseline_inputs["input_ids"].shape[-1]
generated_tokens = baseline_output[0, prompt_length:]

baseline_answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
).strip()

print("Pergunta:")
print(test_example["question"])

print("\nResposta de referência:")
print(test_example["answer"])

print("\nResposta do modelo-base:")
print(baseline_answer)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Pergunta:
What is (are) small fiber neuropathy ?

Resposta de referência:
Small fiber neuropathy is a condition characterized by severe pain attacks that typically begin in the feet or hands. As a person ages, the pain attacks can affect other regions. Some people initially experience a more generalized, whole-body pain. The attacks usually consist of pain described as stabbing or burning, or abnormal skin sensations such as tingling or itchiness. In some individuals, the pain is more severe during times of rest or at night. The signs and symptoms of small fiber neuropathy usually begin in adolescence to mid-adulthood. Individuals with small fiber neuropathy cannot feel pain that is concentrated in a very small area, such as the prick of a pin. However, they have an increased sensitivity to pain in general (hyperalgesia) and experience pain from stimulation that typically does not cause pain (hypoesthesia). People affected with this condition may also have a reduced ability to differen

In [23]:
"""
Restaura o modelo para o modo de treinamento e desativa o cache incompatível 
com gradient checkpointing.
"""

model.config.use_cache = False
_ = model.train()

In [24]:
"""
Calcula a quantidade lógica de parâmetros totais e o percentual 
efetivamente treinado pelo LoRA.
"""

trainable_parameters, total_parameters = (
    model.get_nb_trainable_parameters()
)

print(f"Parâmetros treináveis: {trainable_parameters:,}")
print(f"Parâmetros lógicos totais: {total_parameters:,}")
print(
    "Percentual treinável:",
    f"{100 * trainable_parameters / total_parameters:.4f}%",
)

Parâmetros treináveis: 5,636,096
Parâmetros lógicos totais: 1,241,450,496
Percentual treinável: 0.4540%


In [ ]:
"""
NÃO EXECUTAR - PERTENCE AO FINE-TUNING PILOTO
Seleciona deterministicamente 500 exemplos de treino e 100 
de validação para o piloto. 
"""

import numpy as np

PILOT_SEED = 42
PILOT_TRAIN_SIZE = 500
PILOT_VALIDATION_SIZE = 100

rng = np.random.default_rng(PILOT_SEED)

pilot_train_indices = rng.permutation(
    len(training_datasets["train"])
)[:PILOT_TRAIN_SIZE].tolist()

pilot_validation_indices = rng.permutation(
    len(training_datasets["validation"])
)[:PILOT_VALIDATION_SIZE].tolist()

pilot_train = training_datasets["train"].select(
    pilot_train_indices
)

pilot_validation = training_datasets["validation"].select(
    pilot_validation_indices
)

print("Treino piloto:", len(pilot_train))
print("Validação piloto:", len(pilot_validation))
print("Teste usado no treinamento: 0")

In [ ]:
"""
NÃO EXECUTAR - PERTENCE AO FINE-TUNING PILOTO
Registra os primeiros IDs selecionados para demonstrar que o 
conjunto piloto é reproduzível.
"""

pilot_train_ids = raw_datasets["train"].select(
    pilot_train_indices
)["id"]

pilot_validation_ids = raw_datasets["validation"].select(
    pilot_validation_indices
)["id"]

print("Primeiros IDs de treino:", pilot_train_ids[:5])
print(
    "Primeiros IDs de validação:",
    pilot_validation_ids[:5],
)

In [ ]:
"""
NÃO EXECUTAR - PERTENCE AO FINE-TUNING PILOTO
Define os hiperparâmetros do piloto e cria o SFTTrainer com treino 
supervisionado somente nas respostas.
"""

from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = (
    "/kaggle/working/"
    "medassist-llama32-1b-qlora-pilot"
)

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,

    max_length=768,
    completion_only_loss=True,
    packing=False,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    logging_strategy="steps",
    logging_steps=10,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    report_to="none",
    seed=PILOT_SEED,
    data_seed=PILOT_SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=pilot_train,
    eval_dataset=pilot_validation,
    processing_class=tokenizer,
)

print("Trainer configurado.")
print("Diretório de saída:", OUTPUT_DIR)

In [25]:
"""
Configura o treinamento QLoRA definitivo com todos os registros de
treino e validação. A configuração usa uma época, contexto de 1024
tokens e salva checkpoints periódicos com base na loss de validação.
"""

from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = (
    "/kaggle/working/"
    "medassist-llama32-1b-qlora-final"
)

training_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Executa uma passagem completa pelos 4.795 exemplos.
    num_train_epochs=1,

    # O batch pequeno reduz o consumo de VRAM da Tesla T4.
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # Simula um batch efetivo de oito exemplos.
    gradient_accumulation_steps=8,

    # Taxa menor que a do piloto para maior estabilidade
    # durante o treinamento com o conjunto completo.
    learning_rate=1e-4,
    lr_scheduler_type="cosine",

    # Aproximadamente 5% dos 600 passos estimados.
    warmup_steps=30,

    # Preserva aproximadamente 97% dos exemplos sem truncamento.
    max_length=1024,

    # Calcula a loss somente sobre a resposta do assistente.
    completion_only_loss=True,
    packing=False,

    # Configuração compatível com a Tesla T4.
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    # Registra a evolução do treino a cada 25 passos.
    logging_strategy="steps",
    logging_steps=25,

    # Avalia e salva três vezes durante a época.
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # Ao final, recupera o checkpoint com menor eval_loss.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=FINAL_SEED,
    data_seed=FINAL_SEED,
)

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=final_train,
    eval_dataset=final_validation,
    processing_class=tokenizer,
)

print("Trainer definitivo configurado.")
print("Diretório de saída:", OUTPUT_DIR)
print("Registros de treino:", len(trainer.train_dataset))
print("Registros de validação:", len(trainer.eval_dataset))
print("Comprimento máximo:", training_config.max_length)
print(
    "Batch efetivo:",
    (
        training_config.per_device_train_batch_size
        * training_config.gradient_accumulation_steps
    ),
)

Tokenizing train dataset:   0%|          | 0/4795 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4795 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4795 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4795 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/603 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/603 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/603 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/603 [00:00<?, ? examples/s]

Trainer definitivo configurado.
Diretório de saída: /kaggle/working/medassist-llama32-1b-qlora-final
Registros de treino: 4795
Registros de validação: 603
Comprimento máximo: 1024
Batch efetivo: 8


In [26]:
"""
Cria o primeiro batch do treinamento definitivo e verifica
se os campos, dimensões e tokens supervisionados estão corretos.
"""

train_dataloader = trainer.get_train_dataloader()
first_batch = next(iter(train_dataloader))

print("Campos do batch:", list(first_batch.keys()))
print(
    "Formato de input_ids:",
    tuple(first_batch["input_ids"].shape),
)
print(
    "Formato de labels:",
    tuple(first_batch["labels"].shape),
)

supervised_mask = first_batch["labels"] != -100
supervised_tokens = int(supervised_mask.sum())

print(
    "Tokens supervisionados no primeiro batch:",
    supervised_tokens,
)

assert supervised_tokens > 0, (
    "O batch não contém tokens supervisionados."
)

Campos do batch: ['input_ids', 'labels', 'attention_mask']
Formato de input_ids: (1, 266)
Formato de labels: (1, 266)
Tokens supervisionados no primeiro batch: 183


In [27]:
"""
Decodifica os tokens supervisionados do primeiro batch para confirmar
que somente a resposta do assistente será usada no cálculo da loss.
"""

first_labels = first_batch["labels"][0]

answer_token_ids = first_labels[
    first_labels != -100
]

supervised_text = tokenizer.decode(
    answer_token_ids,
    skip_special_tokens=True,
)

print("Trecho supervisionado:")
print(supervised_text[:1500])

assert supervised_text.strip(), (
    "O texto supervisionado está vazio."
)

Trecho supervisionado:
In most people with Ollier disease, the disorder is caused by mutations in the IDH1 or IDH2 gene. These genes provide instructions for making enzymes called isocitrate dehydrogenase 1 and isocitrate dehydrogenase 2, respectively. These enzymes convert a compound called isocitrate to another compound called 2-ketoglutarate. This reaction also produces a molecule called NADPH, which is necessary for many cellular processes. IDH1 or IDH2 gene mutations cause the enzyme produced from the respective gene to take on a new, abnormal function. Although these mutations have been found in some cells of enchondromas in people with Ollier disease, the relationship between the mutations and the signs and symptoms of the disorder is not well understood. Mutations in other genes may also account for some cases of Ollier disease.


In [32]:
"""
Converte todos os parâmetros treináveis do LoRA para FP32,
confirma que o otimizador ainda não foi criado e calcula
a quantidade esperada de passos do treinamento definitivo.
"""

import math
import torch

from collections import Counter

trainer.model.zero_grad(set_to_none=True)

optimizer_already_created = trainer.optimizer is not None

print(
    "Otimizador já criado:",
    optimizer_already_created,
)

if optimizer_already_created:
    raise RuntimeError(
        "O otimizador foi criado antes da conversão para FP32."
    )

trainable_dtypes_before = Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
)

print(
    "Tipos treináveis antes da conversão:",
    trainable_dtypes_before,
)

converted_parameters = 0

for name, parameter in trainer.model.named_parameters():
    if (
        parameter.requires_grad
        and parameter.dtype != torch.float32
    ):
        parameter.data = parameter.data.to(
            dtype=torch.float32
        )
        converted_parameters += 1

trainable_dtypes_after = Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
)

micro_batches_per_epoch = len(train_dataloader)

optimizer_steps_per_epoch = math.ceil(
    micro_batches_per_epoch
    / training_config.gradient_accumulation_steps
)

estimated_total_steps = math.ceil(
    optimizer_steps_per_epoch
    * training_config.num_train_epochs
)

effective_batch_size = (
    training_config.per_device_train_batch_size
    * training_config.gradient_accumulation_steps
)

print(
    "Parâmetros convertidos para FP32:",
    converted_parameters,
)

print(
    "Tipos treináveis depois da conversão:",
    trainable_dtypes_after,
)

print(
    "Micro-batches por época:",
    micro_batches_per_epoch,
)

print(
    "Passos do otimizador por época:",
    optimizer_steps_per_epoch,
)

print(
    "Total estimado de passos:",
    estimated_total_steps,
)

print(
    "Batch efetivo:",
    effective_batch_size,
)

assert converted_parameters == 224

assert set(
    trainable_dtypes_after.keys()
) == {"torch.float32"}

assert trainer.optimizer is None
assert estimated_total_steps == 600

print(
    "Trainer validado e pronto para o "
    "treinamento definitivo."
)

Otimizador já criado: False
Tipos treináveis antes da conversão: Counter({'torch.bfloat16': 224})
Parâmetros convertidos para FP32: 224
Tipos treináveis depois da conversão: Counter({'torch.float32': 224})
Micro-batches por época: 4795
Passos do otimizador por época: 600
Total estimado de passos: 600
Batch efetivo: 8
Trainer validado e pronto para o treinamento definitivo.


In [33]:
"""
Executa o treinamento QLoRA definitivo, mede o tempo total e o
pico de memória da GPU e apresenta as métricas produzidas pelo Trainer.
"""

import time
import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_started_at = time.perf_counter()

train_result = trainer.train()

training_elapsed_seconds = (
    time.perf_counter() - training_started_at
)

peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

print(
    "Tempo total de treinamento:",
    f"{training_elapsed_seconds / 60:.2f} minutos",
)

print(
    "Pico de VRAM:",
    f"{peak_gpu_memory_gb:.2f} GB",
)

print(
    "Passos concluídos:",
    trainer.state.global_step,
)

print(
    "Melhor checkpoint:",
    trainer.state.best_model_checkpoint,
)

print(
    "Melhor eval_loss:",
    trainer.state.best_metric,
)

print("Métricas finais:")

for metric_name, metric_value in train_result.metrics.items():
    print(
        f"{metric_name}:",
        metric_value,
    )

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.322608,1.376935,1.409560,536162.000000,0.678247
400,1.260524,1.332324,1.355325,1078374.000000,0.685335
600,1.356919,1.321857,1.340589,1608261.000000,0.687311


Tempo total de treinamento: 43.38 minutos
Pico de VRAM: 2.58 GB
Passos concluídos: 600
Melhor checkpoint: /kaggle/working/medassist-llama32-1b-qlora-final/checkpoint-600
Melhor eval_loss: 1.3218574523925781
Métricas finais:
train_runtime: 2602.4192
train_samples_per_second: 1.843
train_steps_per_second: 0.231
total_flos: 9444823515721728.0
train_loss: 1.3579773648579916
epoch: 1.0


In [34]:
"""
Salva o adapter QLoRA definitivo, o tokenizer, as métricas,
o histórico do Trainer e um resumo reproduzível do treinamento.
Nenhuma chave ou token de autenticação é incluído nos artefatos.
"""

import json
from datetime import datetime, timezone
from pathlib import Path

FINAL_ADAPTER_DIR = (
    Path(OUTPUT_DIR) / "final_adapter"
)

FINAL_ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

trainer.save_model(
    str(FINAL_ADAPTER_DIR)
)

tokenizer.save_pretrained(
    str(FINAL_ADAPTER_DIR)
)

trainer.save_state()

trainer.save_metrics(
    "train",
    train_result.metrics,
)

log_history_file = (
    Path(OUTPUT_DIR) / "trainer_log_history.json"
)

with log_history_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        trainer.state.log_history,
        file,
        ensure_ascii=False,
        indent=2,
    )

evaluation_records = [
    record
    for record in trainer.state.log_history
    if "eval_loss" in record
]

run_summary = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "base_model": MODEL_ID,
    "training_method": "QLoRA",
    "quantization": {
        "load_in_4bit": True,
        "quant_type": "nf4",
        "double_quantization": True,
        "compute_dtype": "float16",
    },
    "lora": {
        "rank": 8,
        "alpha": 16,
        "dropout": 0.05,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    },
    "dataset": {
        "train_records": len(final_train),
        "validation_records": len(final_validation),
        "test_records_used_for_training": 0,
    },
    "training": {
        "seed": FINAL_SEED,
        "epochs": training_config.num_train_epochs,
        "max_length": training_config.max_length,
        "learning_rate": training_config.learning_rate,
        "gradient_accumulation_steps": (
            training_config.gradient_accumulation_steps
        ),
        "effective_batch_size": 8,
        "completed_steps": trainer.state.global_step,
        "runtime_seconds": train_result.metrics[
            "train_runtime"
        ],
        "train_loss": train_result.metrics[
            "train_loss"
        ],
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
    },
    "evaluation": {
        "best_checkpoint": (
            trainer.state.best_model_checkpoint
        ),
        "best_eval_loss": trainer.state.best_metric,
        "records": evaluation_records,
    },
}

summary_file = (
    Path(OUTPUT_DIR) / "training_summary.json"
)

with summary_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Adapter salvo em:", FINAL_ADAPTER_DIR)
print("Histórico salvo em:", log_history_file)
print("Resumo salvo em:", summary_file)

print("\nArquivos do adapter:")

for file_path in sorted(
    FINAL_ADAPTER_DIR.iterdir()
):
    print(
        file_path.name,
        f"({file_path.stat().st_size / 1024**2:.2f} MB)",
    )

Adapter salvo em: /kaggle/working/medassist-llama32-1b-qlora-final/final_adapter
Histórico salvo em: /kaggle/working/medassist-llama32-1b-qlora-final/trainer_log_history.json
Resumo salvo em: /kaggle/working/medassist-llama32-1b-qlora-final/training_summary.json

Arquivos do adapter:
README.md (0.00 MB)
adapter_config.json (0.00 MB)
adapter_model.safetensors (21.53 MB)
chat_template.jinja (0.00 MB)
tokenizer.json (16.41 MB)
tokenizer_config.json (0.00 MB)
training_args.bin (0.01 MB)


In [35]:
"""
Executa uma comparação controlada entre o modelo-base e o modelo
com o adapter definitivo nas mesmas três perguntas usadas no piloto.
As duas versões utilizam os mesmos parâmetros contra repetição, e
os resultados são salvos em JSONL para avaliação posterior.
"""

import json
import torch

from contextlib import nullcontext
from pathlib import Path

COMPARISON_IDS = [
    "medquad_010cd44e773a9f9e4c8a6021",
    "medquad_029cacaae54982c21d282300",
    "medquad_033c5d4ab877ed553a3de632",
]

comparison_file = (
    Path(OUTPUT_DIR)
    / "post_training_comparison_final.jsonl"
)

test_records_by_id = {
    record["id"]: record
    for record in raw_datasets["test"]
}

missing_ids = [
    record_id
    for record_id in COMPARISON_IDS
    if record_id not in test_records_by_id
]

if missing_ids:
    raise ValueError(
        f"IDs não encontrados no teste: {missing_ids}"
    )

inference_model = trainer.model

inference_model.eval()
inference_model.config.use_cache = True


def generate_medassist_answer(
    record,
    adapter_enabled,
):
    """
    Gera uma resposta usando o mesmo prompt e os mesmos parâmetros.
    A única diferença é a ativação ou desativação do adapter LoRA.
    """

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": record["question"].strip(),
        },
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    model_inputs = {
        name: tensor.to("cuda:0")
        for name, tensor in model_inputs.items()
    }

    adapter_context = (
        nullcontext()
        if adapter_enabled
        else inference_model.disable_adapter()
    )

    with adapter_context:
        with torch.inference_mode():
            generated_output = inference_model.generate(
                **model_inputs,
                max_new_tokens=384,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=4,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    prompt_length = model_inputs[
        "input_ids"
    ].shape[-1]

    generated_tokens = generated_output[
        0,
        prompt_length:,
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


comparison_results = []

for record_id in COMPARISON_IDS:
    record = test_records_by_id[record_id]

    print("\n" + "=" * 80)
    print("ID:", record_id)
    print("Pergunta:", record["question"])

    base_answer = generate_medassist_answer(
        record=record,
        adapter_enabled=False,
    )

    adapted_answer = generate_medassist_answer(
        record=record,
        adapter_enabled=True,
    )

    result = {
        "id": record["id"],
        "question": record["question"],
        "reference_answer": record["answer"],
        "base_answer": base_answer,
        "adapted_answer": adapted_answer,
        "base_model": MODEL_ID,
        "adapter": str(FINAL_ADAPTER_DIR),
        "generation": {
            "max_new_tokens": 384,
            "do_sample": False,
            "repetition_penalty": 1.1,
            "no_repeat_ngram_size": 4,
        },
    }

    comparison_results.append(result)

    print("\nResposta do modelo-base:")
    print(base_answer)

    print("\nResposta do modelo ajustado:")
    print(adapted_answer)

with comparison_file.open(
    "w",
    encoding="utf-8",
) as file:
    for result in comparison_results:
        file.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

different_answers = sum(
    result["base_answer"]
    != result["adapted_answer"]
    for result in comparison_results
)

print("\n" + "=" * 80)
print("Comparações concluídas:", len(comparison_results))
print("Respostas diferentes:", different_answers)
print("Arquivo salvo em:", comparison_file)

assert len(comparison_results) == 3
assert different_answers == 3


ID: medquad_010cd44e773a9f9e4c8a6021
Pergunta: What is (are) small fiber neuropathy ?

Resposta do modelo-base:
Small fiber neuropathy (SFN) is a condition that affects the nerves in the body, causing damage to the small nerve fibers. These tiny nerve fibers play a crucial role in transmitting signals between the brain and the rest of the body.

In SFN, the small nerve cells are damaged, leading to a loss of sensation in the affected areas. This can cause a range of symptoms, including:

* Burning or stinging sensations in the skin
* Tingling or numbness in the hands and feet
* Painful or burning sensations when walking or standing for long periods
* Weakness or fatigue in the muscles
* Numbness or tingling in the face, arms, or legs

The exact causes of SFN are still not fully understood, but it is believed to be related to factors such as:

* Genetics
* Environmental toxins (e.g., pesticides, heavy metals)
* Infections (e.g. Lyme disease, HIV)
* Autoimmune disorders (e.g.? Systemic 

In [36]:
"""
Cria um pacote versionado contendo o adapter definitivo, tokenizer,
métricas, logs e comparação pós-treinamento. Também calcula hashes
SHA-256 para verificar posteriormente a integridade dos artefatos.
Os checkpoints intermediários não são incluídos no ZIP.
"""

import hashlib
import json
import shutil

from datetime import datetime, timezone
from pathlib import Path

ARTIFACT_VERSION = "1.0.0"

PACKAGE_NAME = (
    f"medassist-llama32-1b-qlora-"
    f"adapter-v{ARTIFACT_VERSION}"
)

PACKAGE_DIR = (
    Path("/kaggle/working") / PACKAGE_NAME
)

PACKAGE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PACKAGE_ADAPTER_DIR = (
    PACKAGE_DIR / "adapter"
)

shutil.copytree(
    FINAL_ADAPTER_DIR,
    PACKAGE_ADAPTER_DIR,
    dirs_exist_ok=True,
)

evidence_files = [
    Path(OUTPUT_DIR) / "training_summary.json",
    Path(OUTPUT_DIR) / "trainer_log_history.json",
    Path(OUTPUT_DIR) / "post_training_comparison_final.jsonl",
    Path(OUTPUT_DIR) / "train_results.json",
    Path(OUTPUT_DIR) / "trainer_state.json",
]

copied_evidence_files = []

for source_file in evidence_files:
    if source_file.exists():
        destination_file = (
            PACKAGE_DIR / source_file.name
        )

        shutil.copy2(
            source_file,
            destination_file,
        )

        copied_evidence_files.append(
            destination_file.name
        )


def calculate_sha256(file_path):
    """
    Calcula o hash SHA-256 de um arquivo sem carregá-lo
    integralmente na memória.
    """

    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


manifest_entries = []

for artifact_file in sorted(
    path
    for path in PACKAGE_DIR.rglob("*")
    if path.is_file()
):
    manifest_entries.append({
        "path": str(
            artifact_file.relative_to(PACKAGE_DIR)
        ),
        "size_bytes": artifact_file.stat().st_size,
        "sha256": calculate_sha256(
            artifact_file
        ),
    })

manifest = {
    "artifact_name": PACKAGE_NAME,
    "artifact_version": ARTIFACT_VERSION,
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "base_model": MODEL_ID,
    "training_method": "QLoRA",
    "adapter_format": "PEFT",
    "files": manifest_entries,
}

manifest_file = (
    PACKAGE_DIR / "artifact_manifest.json"
)

with manifest_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

archive_path = Path(
    shutil.make_archive(
        str(
            Path("/kaggle/working")
            / PACKAGE_NAME
        ),
        "zip",
        root_dir=PACKAGE_DIR,
    )
)

archive_sha256 = calculate_sha256(
    archive_path
)

hash_file = Path(
    str(archive_path) + ".sha256.txt"
)

hash_file.write_text(
    f"{archive_sha256}  {archive_path.name}\n",
    encoding="utf-8",
)

print("Versão do artefato:", ARTIFACT_VERSION)
print("Arquivos de evidência:", copied_evidence_files)
print("Quantidade de arquivos no manifesto:", len(manifest_entries))
print("ZIP criado em:", archive_path)
print(
    "Tamanho do ZIP:",
    f"{archive_path.stat().st_size / 1024**2:.2f} MB",
)
print("SHA-256 do ZIP:", archive_sha256)
print("Arquivo de hash:", hash_file)

Versão do artefato: 1.0.0
Arquivos de evidência: ['training_summary.json', 'trainer_log_history.json', 'post_training_comparison_final.jsonl', 'train_results.json', 'trainer_state.json']
Quantidade de arquivos no manifesto: 12
ZIP criado em: /kaggle/working/medassist-llama32-1b-qlora-adapter-v1.0.0.zip
Tamanho do ZIP: 22.43 MB
SHA-256 do ZIP: 116fb47389f0e0742375753755f94b338d9fcfa8ba7aad93fc8e9c265996bac0
Arquivo de hash: /kaggle/working/medassist-llama32-1b-qlora-adapter-v1.0.0.zip.sha256.txt


In [3]:
"""
Localiza diretamente os arquivos do adapter no dataset que o Kaggle
descompactou automaticamente durante o upload. Também confirma que
a configuração e os pesos pertencem ao mesmo diretório.
"""

from pathlib import Path

PACKAGE_INPUT_DIR = Path(
    "/kaggle/input/datasets/dharmasistemas/"
    "medassist-qlora-adapter-v1-zip"
)

print(
    "Diretório de entrada existe:",
    PACKAGE_INPUT_DIR.exists(),
)

if not PACKAGE_INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Diretório não encontrado: {PACKAGE_INPUT_DIR}"
    )

print("\nArquivos disponíveis:")

for file_path in sorted(
    path
    for path in PACKAGE_INPUT_DIR.rglob("*")
    if path.is_file()
):
    print(
        file_path.relative_to(PACKAGE_INPUT_DIR),
        f"({file_path.stat().st_size / 1024**2:.2f} MB)",
    )

config_candidates = list(
    PACKAGE_INPUT_DIR.rglob(
        "adapter_config.json"
    )
)

weights_candidates = list(
    PACKAGE_INPUT_DIR.rglob(
        "adapter_model.safetensors"
    )
)

print(
    "\nConfigurações encontradas:",
    len(config_candidates),
)

print(
    "Arquivos de pesos encontrados:",
    len(weights_candidates),
)

if len(config_candidates) != 1:
    raise RuntimeError(
        "Era esperado exatamente um adapter_config.json."
    )

if len(weights_candidates) != 1:
    raise RuntimeError(
        "Era esperado exatamente um adapter_model.safetensors."
    )

adapter_config_file = config_candidates[0]
adapter_weights_file = weights_candidates[0]

if (
    adapter_config_file.parent
    != adapter_weights_file.parent
):
    raise RuntimeError(
        "A configuração e os pesos estão em diretórios diferentes."
    )

ADAPTER_DIR = adapter_config_file.parent

print("\nDiretório do adapter:", ADAPTER_DIR)
print("Configuração:", adapter_config_file)
print("Pesos:", adapter_weights_file)

print(
    "Tamanho dos pesos:",
    f"{adapter_weights_file.stat().st_size / 1024**2:.2f} MB",
)

print("Adapter pronto para conversão.")

Diretório de entrada existe: True

Arquivos disponíveis:
adapter/README.md (0.00 MB)
adapter/adapter_config.json (0.00 MB)
adapter/adapter_model.safetensors (21.53 MB)
adapter/chat_template.jinja (0.00 MB)
adapter/tokenizer.json (16.41 MB)
adapter/tokenizer_config.json (0.00 MB)
adapter/training_args.bin (0.01 MB)
artifact_manifest.json (0.00 MB)
post_training_comparison_final.jsonl (0.01 MB)
train_results.json (0.00 MB)
trainer_log_history.json (0.01 MB)
trainer_state.json (0.01 MB)
training_summary.json (0.00 MB)

Configurações encontradas: 1
Arquivos de pesos encontrados: 1

Diretório do adapter: /kaggle/input/datasets/dharmasistemas/medassist-qlora-adapter-v1-zip/adapter
Configuração: /kaggle/input/datasets/dharmasistemas/medassist-qlora-adapter-v1-zip/adapter/adapter_config.json
Pesos: /kaggle/input/datasets/dharmasistemas/medassist-qlora-adapter-v1-zip/adapter/adapter_model.safetensors
Tamanho dos pesos: 21.53 MB
Adapter pronto para conversão.


In [4]:
"""
Baixa uma cópia rasa do repositório oficial llama.cpp, localiza
o conversor de adapters LoRA e registra o commit utilizado para
que a conversão possa ser reproduzida posteriormente.
"""

import subprocess

from pathlib import Path

LLAMA_CPP_DIR = Path(
    "/kaggle/working/llama.cpp"
)

if LLAMA_CPP_DIR.exists():
    print(
        "Repositório llama.cpp já está disponível."
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/ggml-org/llama.cpp.git",
            str(LLAMA_CPP_DIR),
        ],
        check=True,
    )

CONVERTER_SCRIPT = (
    LLAMA_CPP_DIR
    / "convert_lora_to_gguf.py"
)

if not CONVERTER_SCRIPT.exists():
    raise FileNotFoundError(
        f"Conversor não encontrado: {CONVERTER_SCRIPT}"
    )

commit_hash = subprocess.check_output(
    [
        "git",
        "-C",
        str(LLAMA_CPP_DIR),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

print("Conversor:", CONVERTER_SCRIPT)
print("Commit do llama.cpp:", commit_hash)
print("Conversor pronto.")

Cloning into '/kaggle/working/llama.cpp'...


Conversor: /kaggle/working/llama.cpp/convert_lora_to_gguf.py
Commit do llama.cpp: 67672dc5b76f8bc17785a19d3dc6d1463fc2902c
Conversor pronto.


Updating files: 100% (3548/3548), done.


In [5]:
"""
Recupera o token do Hugging Face armazenado nos Secrets do Kaggle
e o disponibiliza ao conversor sem imprimir seu conteúdo.
O token é necessário para acessar a configuração do Llama 3.2.
"""

import os

from kaggle_secrets import UserSecretsClient

secrets_client = UserSecretsClient()

hf_token = secrets_client.get_secret(
    "HF_TOKEN"
)

if not hf_token:
    raise RuntimeError(
        "O secret HF_TOKEN não está configurado."
    )

os.environ["HF_TOKEN"] = hf_token

print(
    "Token do Hugging Face disponibilizado "
    "sem exibir seu valor."
)

Token do Hugging Face disponibilizado sem exibir seu valor.


In [6]:
"""
Converte o adapter PEFT/Safetensors definitivo para o formato GGUF
em precisão F16, compatível com a instrução ADAPTER do Ollama.
Também valida a assinatura GGUF e calcula o hash SHA-256 do arquivo.
"""

import hashlib
import subprocess
import sys

from pathlib import Path

BASE_MODEL_ID = (
    "meta-llama/Llama-3.2-1B-Instruct"
)

GGUF_ADAPTER_FILE = Path(
    "/kaggle/working/"
    "medassist-llama32-1b-qlora-"
    "adapter-v1.0.0-f16.gguf"
)

conversion_command = [
    sys.executable,
    str(CONVERTER_SCRIPT),
    str(ADAPTER_DIR),
    "--base-model-id",
    BASE_MODEL_ID,
    "--outfile",
    str(GGUF_ADAPTER_FILE),
    "--outtype",
    "f16",
]

print("Iniciando conversão para GGUF F16.")

subprocess.run(
    conversion_command,
    check=True,
)

if not GGUF_ADAPTER_FILE.exists():
    raise FileNotFoundError(
        "O conversor não criou o arquivo GGUF."
    )

with GGUF_ADAPTER_FILE.open("rb") as file:
    gguf_signature = file.read(4)

if gguf_signature != b"GGUF":
    raise RuntimeError(
        f"Assinatura GGUF inválida: {gguf_signature!r}"
    )

sha256 = hashlib.sha256()

with GGUF_ADAPTER_FILE.open("rb") as file:
    for block in iter(
        lambda: file.read(1024 * 1024),
        b"",
    ):
        sha256.update(block)

gguf_sha256 = sha256.hexdigest()

hash_file = Path(
    str(GGUF_ADAPTER_FILE) + ".sha256.txt"
)

hash_file.write_text(
    (
        f"{gguf_sha256}  "
        f"{GGUF_ADAPTER_FILE.name}\n"
    ),
    encoding="utf-8",
)

print("Conversão concluída.")
print("Arquivo:", GGUF_ADAPTER_FILE)
print(
    "Tamanho:",
    f"{GGUF_ADAPTER_FILE.stat().st_size / 1024**2:.2f} MB",
)
print("SHA-256:", gguf_sha256)
print("Arquivo de hash:", hash_file)

Iniciando conversão para GGUF F16.


INFO:lora-to-gguf:Loading base model from Hugging Face: meta-llama/Llama-3.2-1B-Instruct
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:lora-to-gguf:Using model architecture: LlamaForCausalLM
INFO:hf-to-gguf:Using remote model with HuggingFace id: meta-llama/Llama-3.2-1B-Instruct
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:lora-to-gguf:Exporting model...
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_a, torch.float32 --> F16, shape = {8192, 8}
INFO:hf-to-gguf:blk.0.ffn_down.weight.lora_b, torch.float32 --> F16, shape = {8, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_a, torch.float32 --> F16, shape = {2048, 8}
INFO:hf-to-gguf:blk.0.ffn_gate.weight.lora_b, torch.float32 --> F16, shape = {8, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight.lora_a,  torch

Conversão concluída.
Arquivo: /kaggle/working/medassist-llama32-1b-qlora-adapter-v1.0.0-f16.gguf
Tamanho: 10.77 MB
SHA-256: 4ef5939ec02b27cd2296b713e6ad2154189d108b61623a85a813ef2010e8f189
Arquivo de hash: /kaggle/working/medassist-llama32-1b-qlora-adapter-v1.0.0-f16.gguf.sha256.txt


In [ ]:
"""
Inspeciona um batch e confirma dimensões, campos e quantidade 
de tokens supervisionados.
"""

print(
    "Registros de treino após preparação:",
    len(trainer.train_dataset),
)

print(
    "Registros de validação após preparação:",
    len(trainer.eval_dataset),
)

train_dataloader = trainer.get_train_dataloader()
batch = next(iter(train_dataloader))

print("Campos do batch:", list(batch.keys()))
print("Formato de input_ids:", tuple(batch["input_ids"].shape))
print("Formato de labels:", tuple(batch["labels"].shape))

supervised_mask = batch["labels"] != -100
supervised_tokens = int(supervised_mask.sum())

print("Tokens supervisionados no batch:", supervised_tokens)

In [ ]:
"""
Decodifica os labels supervisionados para confirmar que somente a
resposta contribui para a loss.
"""

first_labels = batch["labels"][0]
answer_token_ids = first_labels[first_labels != -100]

decoded_supervised_text = tokenizer.decode(
    answer_token_ids,
    skip_special_tokens=True,
)

print("Trecho supervisionado:")
print(decoded_supervised_text[:1000])

In [ ]:
"""
Calcula micro-batches, batch efetivo e quantidade esperada de 
passos do otimizador.
"""

import math

micro_batches_per_epoch = len(train_dataloader)

optimizer_steps_per_epoch = math.ceil(
    micro_batches_per_epoch
    / training_config.gradient_accumulation_steps
)

estimated_total_steps = (
    optimizer_steps_per_epoch
    * int(training_config.num_train_epochs)
)

print("Micro-batches por época:", micro_batches_per_epoch)
print(
    "Acumulação de gradientes:",
    training_config.gradient_accumulation_steps,
)
print(
    "Passos do otimizador por época:",
    optimizer_steps_per_epoch,
)
print("Total estimado de passos:", estimated_total_steps)
print("Batch efetivo: 8 exemplos")

In [ ]:
"""
Verifica o dtype dos adaptadores após a criação do Trainer e confirma 
que nenhum passo foi executado.
"""

from collections import Counter

trainable_dtypes_before = Counter(
    str(parameter.dtype)
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Tipos treináveis antes da correção:")
print(trainable_dtypes_before)

print("Passos concluídos:", trainer.state.global_step)

In [ ]:
"""
Converte os adaptadores mantidos pelo Trainer de BF16 para FP32 
antes de criar o otimizador.
"""

from collections import Counter

trainer.model.zero_grad(set_to_none=True)

print(
    "Otimizador criado antes da correção:",
    trainer.optimizer is not None,
)

converted_parameters = 0

for parameter in trainer.model.parameters():
    if parameter.requires_grad:
        if parameter.dtype != torch.float32:
            parameter.data = parameter.data.to(torch.float32)
            converted_parameters += 1

trainable_dtypes_after = Counter(
    str(parameter.dtype)
    for parameter in trainer.model.parameters()
    if parameter.requires_grad
)

print(
    "Tensores treináveis convertidos:",
    converted_parameters,
)

print(
    "Tipos treináveis depois da correção:",
    trainable_dtypes_after,
)

assert trainer.optimizer is None
assert set(trainable_dtypes_after) == {"torch.float32"}

print("Trainer pronto para iniciar o treinamento.")

In [ ]:
"""
Executa o treinamento piloto e registra tempo total, 
pico de VRAM e métricas finais.
"""

import time

torch.cuda.reset_peak_memory_stats()

training_started_at = time.perf_counter()

train_result = trainer.train()

training_elapsed_seconds = (
    time.perf_counter() - training_started_at
)

peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

print(
    f"Tempo total: "
    f"{training_elapsed_seconds / 60:.2f} minutos"
)

print(f"Pico de VRAM: {peak_gpu_memory_gb:.2f} GB")

print("Métricas finais:")
print(train_result.metrics)

In [ ]:
"""
Salva o adaptador LoRA, o tokenizer, o estado do Trainer e as 
métricas no diretório de saída.
"""

from pathlib import Path
import os

os.environ["HF_TOKEN"] = hf_token

FINAL_ADAPTER_DIR = (
    Path(OUTPUT_DIR) / "final_adapter"
)

trainer.save_model(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
trainer.save_state()
trainer.save_metrics("train", train_result.metrics)

print("Adaptador final:", FINAL_ADAPTER_DIR)

print("\nArquivos do adaptador:")
for path in sorted(FINAL_ADAPTER_DIR.iterdir()):
    print(
        path.name,
        f"({path.stat().st_size / 1024**2:.2f} MB)",
    )

In [ ]:
"""
Exporta o histórico de treinamento e identifica a 
última avaliação registrada.
"""

import json
from pathlib import Path

log_history_file = (
    Path(OUTPUT_DIR) / "trainer_log_history.json"
)

with log_history_file.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        trainer.state.log_history,
        file,
        ensure_ascii=False,
        indent=2,
    )

evaluation_records = [
    record
    for record in trainer.state.log_history
    if "eval_loss" in record
]

print("Avaliações registradas:", len(evaluation_records))

if evaluation_records:
    print("Última avaliação:")
    print(evaluation_records[-1])

print("Histórico salvo em:", log_history_file)

In [ ]:
"""
Compacta checkpoints, adaptador e métricas em um arquivo ZIP para download.
"""

import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/"
    "medassist-llama32-1b-qlora-pilot",
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Arquivo para download:")
print(archive_path)

In [ ]:
# Localiza o adaptador anexado; se estiver compactado, extrai somente o necessário.
from pathlib import Path
from zipfile import ZipFile

INPUT_ROOT = Path("/kaggle/input")
EXTRACT_ROOT = Path("/kaggle/working/qlora_artifacts")

adapter_configs = list(
    INPUT_ROOT.rglob("final_adapter/adapter_config.json")
)

if adapter_configs:
    ADAPTER_DIR = adapter_configs[0].parent
else:
    zip_files = list(
        INPUT_ROOT.rglob(
            "medassist-llama32-1b-qlora-pilot.zip"
        )
    )

    if not zip_files:
        raise FileNotFoundError(
            "O ZIP do adaptador não foi encontrado."
        )

    EXTRACT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    with ZipFile(zip_files[0]) as archive:
        members = [
            member
            for member in archive.infolist()
            if member.filename.startswith("final_adapter/")
        ]

        if not members:
            raise RuntimeError(
                "O ZIP não contém o diretório final_adapter."
            )

        archive.extractall(
            EXTRACT_ROOT,
            members=members,
        )

    ADAPTER_DIR = EXTRACT_ROOT / "final_adapter"

print("Adaptador localizado em:", ADAPTER_DIR)

for path in sorted(ADAPTER_DIR.iterdir()):
    print(path.name)

In [ ]:
"""
Carrega o Llama em 4 bits e aplica o adaptador LoRA treinado.
"""
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
tokenizer.clean_up_tokenization_spaces = False

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

model.eval()
model.config.use_cache = True

print("Modelo-base:", MODEL_ID)
print("Adaptador:", ADAPTER_DIR)
print("Dispositivo:", next(model.parameters()).device)
print("Adapter ativo:", model.active_adapter)

In [ ]:
"""
Carrega exemplos não utilizados no treinamento para a comparação final.
"""

from datasets import load_dataset

DATA_DIR = (
    "/kaggle/input/datasets/"
    "dharmasistemas/medassist-medquad-splits"
)

test_dataset = load_dataset(
    "json",
    data_files={
        "test": f"{DATA_DIR}/test.jsonl",
    },
    split="test",
)

print("Exemplos de teste:", len(test_dataset))

In [ ]:
"""
Gera respostas determinísticas com ou sem o adaptador LoRA ativo.
"""

from contextlib import nullcontext

SYSTEM_PROMPT = (
    "You are MedAssist, an educational medical information assistant. "
    "Answer the user's medical question clearly and objectively. "
    "Do not present the answer as a medical diagnosis or replace professional care."
)


def generate_answer(question, adapter_enabled):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question.strip(),
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    inputs = {
        name: tensor.to("cuda:0")
        for name, tensor in inputs.items()
    }

    adapter_context = (
        nullcontext()
        if adapter_enabled
        else model.disable_adapter()
    )

    with adapter_context:
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    prompt_length = inputs["input_ids"].shape[-1]
    generated_tokens = output[0, prompt_length:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

In [ ]:
"""
Compara o modelo-base e o modelo ajustado em três exemplos não vistos no treino.
"""

comparison_indices = [0, 1, 2]
comparison_results = []

for index in comparison_indices:
    example = test_dataset[index]
    question = example["question"]

    base_answer = generate_answer(
        question,
        adapter_enabled=False,
    )

    adapted_answer = generate_answer(
        question,
        adapter_enabled=True,
    )

    result = {
        "id": example["id"],
        "question": question,
        "reference_answer": example["answer"],
        "base_answer": base_answer,
        "adapted_answer": adapted_answer,
    }

    comparison_results.append(result)

    print("\n" + "=" * 80)
    print("ID:", example["id"])
    print("\nPERGUNTA:")
    print(question)
    print("\nREFERÊNCIA:")
    print(example["answer"])
    print("\nMODELO-BASE:")
    print(base_answer)
    print("\nMODELO AJUSTADO:")
    print(adapted_answer)

In [ ]:
"""
Salva as respostas comparativas como evidência do piloto.
"""

import json

comparison_file = Path(
    "/kaggle/working/"
    "post_training_comparison.jsonl"
)

with comparison_file.open(
    "w",
    encoding="utf-8",
) as file:
    for result in comparison_results:
        file.write(
            json.dumps(
                result,
                ensure_ascii=False,
            )
            + "\n"
        )

print("Comparação salva em:", comparison_file)